In [ ]:
#Setup e Caricamento del File

!pip install pandas scipy

from google.colab import files
import json
import pandas as pd
import scipy.stats as st
import numpy as np
import io

#Carica il tuo file JSON
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

Saving resultsHistogram.json to resultsHistogram.json


In [ ]:
#Estrazione, Trasformazione e Caricamento

#Caricamento del file JSON in memoria
data = json.load(io.BytesIO(uploaded[file_name]))

all_stats_data = []

# Dizionario per tradurre i nomi criptici (basato sull'omnetpp.ini)
PARAM_MAP = {
    '$0': 'm_SS1',
    '$1': 'm_SS2',
    '$2': 'm_arrivi',
    '$3': 'q_prob_U1',
    '$4': 'p_feedback'
}

for run_id, run_data in data.items():
    attributes = run_data.get('attributes', {})
    config_name = attributes.get('experiment', 'N/A')

    try:
        repetition = int(attributes.get('repetition', -1))
    except ValueError:
        repetition = -1

    itervars_str = attributes.get('iterationvars', '')
    params = {}
    if itervars_str:
        try:
            pairs = itervars_str.split(', ')
            for pair in pairs:
                key, value = pair.split('=')
                col_name = PARAM_MAP.get(key, key)
                params[col_name] = value.replace('s', '')
        except Exception:
            pass

    statistics = run_data.get('statistics', [])

    #Preparazione di una riga di dati per una singola run
    row = {
        'run_id': run_id,
        'Config': config_name,
        'repetition': repetition
    }
    row.update(params)

    found_sojourn_time = False
    for stat in statistics:
        metric_name = stat.get('name')

        #Selezione della statistica necessaria
        if metric_name and metric_name.startswith('sojournTime'):
            found_sojourn_time = True

            #Estrazione 'mean' e 'count' e gestione dei 'null'\
            mean_val = stat.get('mean')
            count_val = stat.get('count')

            # Gestione COUNT: Se manca, è 0
            if count_val is None:
                count_val = 0
            else:
                count_val = int(count_val) # Assicuriamoci sia intero

            # Gestione MEAN: Se count è 0, la media è None
            if count_val == 0:
                mean_val = None
            elif mean_val is None:
                mean_val = 0.0 # Fallback solo se count > 0 ma mean è None (raro)

            #Aggiunta di colonne separate per mean e count
            #'sojournTimeU1_stats:stats' -> 'sojournTimeU1_mean' e 'sojournTimeU1_count'
            base_name = metric_name.replace(':stats', '')

            if mean_val is None:
                row[base_name + '_mean'] = np.nan # Usa NaN (Not a Number) di Numpy
            else:
                row[base_name + '_mean'] = float(mean_val)

            row[base_name + '_count'] = int(count_val)

    #Aggiunta della riga solo se abbiamo trovato almeno una metrica sojournTime
    if found_sojourn_time:
        all_stats_data.append(row)

df = pd.DataFrame(all_stats_data)

display(df.head())

,run_id,Config,repetition,m_SS1,m_SS2,m_arrivi,q_prob_U1,p_feedback,sojournTimeU2_stats_mean,sojournTimeU2_stats_count,sojournTimeU1_stats_mean,sojournTimeU1_stats_count
0,Config1-1107-20251113-21:00:26-17056,Config1,7,2.4,4.0,4.0,0.4,0.8,53000.662264,123.0,288.581102,4022
1,Config1-1108-20251113-21:00:26-17057,Config1,8,2.4,4.0,4.0,0.4,0.8,NaN,0.0,1836.825688,3966
2,Config1-1109-20251113-21:00:26-17058,Config1,9,2.4,4.0,4.0,0.4,0.8,56407.040120,37.0,347.389114,3936
3,Config1-1120-20251113-21:00:27-17069,Config1,0,2.4,4.0,4.0,0.4,0.9,NaN,0.0,18227.147128,1994
4,Config1-1121-20251113-21:00:27-17070,Config1,1,2.4,4.0,4.0,0.4,0.9,NaN,0.0,17951.817627,1976


In [ ]:
#Calcolo Statistiche (Metrica 2: Tempo di Soggiorno)

# 1. Colonne '_mean' e '_count' estratte.
# 2. Calcolo della media di "tutti gli utenti" usando la media pesata per-replica (basata sul count)
# 3. Calcola Stima Puntuale e CI per le 3 metriche finali.

#Definizione delle colonne di configurazione
parametri_configurazione = ['Config', 'm_SS1', 'm_SS2', 'm_arrivi', 'q_prob_U1', 'p_feedback']

# 1. Calcolo metrica 'Tutti gli Utenti' (Media Pesata per-replica)
#Check che le colonne esistano nel caso una run non le abbia registrate
cols_necessarie = ['sojournTimeU1_stats_mean', 'sojournTimeU1_stats_count',
                   'sojournTimeU2_stats_mean', 'sojournTimeU2_stats_count']
for col in cols_necessarie:
    if col not in df.columns:
        df[col] = 0.0

#Calcolo del numeratore (Media * Conteggio) per U1 e U2
num_u1 = df['sojournTimeU1_stats_mean'] * df['sojournTimeU1_stats_count']
num_u2 = df['sojournTimeU2_stats_mean'] * df['sojournTimeU2_stats_count']

#Calcolo del denominatore (Conteggio U1 + Conteggio U2)
den_totale = df['sojournTimeU1_stats_count'] + df['sojournTimeU2_stats_count']

#Calcolo della Media Pesata per "tutti gli utenti"
#Se den_totale è 0, lasciamo che sia NaN (Not a Number)
df['SoggiornoMedio_TuttiUtenti'] = (num_u1 + num_u2).divide(den_totale)
#2. Raggruppamento per le 486 configurazioni uniche
grouped = df.groupby(parametri_configurazione)

#3. Calcolo della Stima Puntuale (Media) e CI
# Valore critico 't' per il 95% di confidenza con (n=20 -> df=19)

#Selezione delle medie delle singole tipologie e la nuova media totale
colonne_output = ['sojournTimeU1_stats_mean', 'sojournTimeU2_stats_mean', 'SoggiornoMedio_TuttiUtenti']
df_per_statistiche = df[parametri_configurazione + ['repetition'] + colonne_output]

#Raggruppamento di nuovo solo sui dati filtrati
grouped = df_per_statistiche.groupby(parametri_configurazione)

#Calcolo delle statistiche aggregate
stima_puntuale = grouped.mean()
deviazione_standard = grouped.std()
n_campioni = grouped.count()

degrees_of_freedom = n_campioni - 1
#Calcola il t critico per ogni riga (vettorizzato)
#np.where per gestire casi limite (es. N=1 -> dof=0) mettendo NaN
t_values = np.where(
    degrees_of_freedom > 0,
    st.t.ppf(0.975, degrees_of_freedom),
    np.nan
)

margine_errore_ci = t_values * (deviazione_standard / np.sqrt(n_campioni))
ci_limite_inferiore = stima_puntuale - margine_errore_ci
ci_limite_superiore = stima_puntuale + margine_errore_ci

#Rimozione della colonna 'repetition'
stima_puntuale = stima_puntuale.drop(columns='repetition')
ci_limite_inferiore = ci_limite_inferiore.drop(columns='repetition')
ci_limite_superiore = ci_limite_superiore.drop(columns='repetition')

#Stima Puntuale (Media delle 20 repliche)
display(stima_puntuale)

#Intervallo di Confidenza (95%) - Limite Inferiore
display(ci_limite_inferiore)

#Intervallo di Confidenza (95%) - Limite Superiore
display(ci_limite_superiore)

sojournTimeU1_stats_mean  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                        16.802813   
                                       0.8                       888.679053   
                                       0.9                     17831.786726   
                             0.6       0.6                        26.061690   
                                       0.8                     14771.799514   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                     14921.062174   
                                       0.9                     23562.967300   
                             0.8       0.6                       763.558734   
                                       0.8                     20776.601663   
                                       0.9                     26223.944752   

                                                   sojournTimeU2_stats_mean  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                     21320.921514   
                                       0.8                     48672.042080   
                                       0.9                              NaN   
                             0.6       0.6                     27049.840601   
                                       0.8                              NaN   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                              NaN   
                                       0.9                              NaN   
                             0.8       0.6                     43745.884384   
                                       0.8                              NaN   
                                       0.9                              NaN   

                                                   SoggiornoMedio_TuttiUtenti  
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                              
Config1 2.4   2.0   4.0      0.4       0.6                        9427.389582  
                                       0.8                        1192.029785  
                                       0.9                                NaN  
                             0.6       0.6                        5632.704637  
                                       0.8                                NaN  
...                                                                       ...  
Config2 3.5   4.0   6.0      0.6       0.8                                NaN  
                                       0.9                                NaN  
                             0.8       0.6                         618.714890  
                                       0.8                                NaN  
                                       0.9                                NaN  

[486 rows x 3 columns]

sojournTimeU1_stats_mean  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                        16.579367   
                                       0.8                       609.002340   
                                       0.9                     17612.826196   
                             0.6       0.6                        25.586288   
                                       0.8                     14498.595488   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                     14714.297872   
                                       0.9                     23385.999152   
                             0.8       0.6                       552.841044   
                                       0.8                     20571.390877   
                                       0.9                     25963.622521   

                                                   sojournTimeU2_stats_mean  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                     21086.108556   
                                       0.8                     42763.878285   
                                       0.9                              NaN   
                             0.6       0.6                     26574.901100   
                                       0.8                              NaN   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                              NaN   
                                       0.9                              NaN   
                             0.8       0.6                     34299.213205   
                                       0.8                              NaN   
                                       0.9                              NaN   

                                                   SoggiornoMedio_TuttiUtenti  
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                              
Config1 2.4   2.0   4.0      0.4       0.6                        9304.780998  
                                       0.8                         955.226921  
                                       0.9                                NaN  
                             0.6       0.6                        5525.685681  
                                       0.8                                NaN  
...                                                                       ...  
Config2 3.5   4.0   6.0      0.6       0.8                                NaN  
                                       0.9                                NaN  
                             0.8       0.6                         419.726979  
                                       0.8                                NaN  
                                       0.9                                NaN  

[486 rows x 3 columns]

sojournTimeU1_stats_mean  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                        17.026259   
                                       0.8                      1168.355767   
                                       0.9                     18050.747256   
                             0.6       0.6                        26.537093   
                                       0.8                     15045.003541   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                     15127.826477   
                                       0.9                     23739.935448   
                             0.8       0.6                       974.276425   
                                       0.8                     20981.812448   
                                       0.9                     26484.266983   

                                                   sojournTimeU2_stats_mean  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                     21555.734471   
                                       0.8                     54580.205876   
                                       0.9                              NaN   
                             0.6       0.6                     27524.780102   
                                       0.8                              NaN   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                              NaN   
                                       0.9                              NaN   
                             0.8       0.6                     53192.555563   
                                       0.8                              NaN   
                                       0.9                              NaN   

                                                   SoggiornoMedio_TuttiUtenti  
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                              
Config1 2.4   2.0   4.0      0.4       0.6                        9549.998167  
                                       0.8                        1428.832649  
                                       0.9                                NaN  
                             0.6       0.6                        5739.723592  
                                       0.8                                NaN  
...                                                                       ...  
Config2 3.5   4.0   6.0      0.6       0.8                                NaN  
                                       0.9                                NaN  
                             0.8       0.6                         817.702801  
                                       0.8                                NaN  
                                       0.9                                NaN  

[486 rows x 3 columns]

In [ ]:
#Download dei Risultati

stima_puntuale.to_csv("metrica2_stima_puntuale.csv")
ci_limite_inferiore.to_csv("metrica2_ci_inferiore.csv")
ci_limite_superiore.to_csv("metrica2_ci_superiore.csv")

files.download("metrica2_stima_puntuale.csv")
files.download("metrica2_ci_inferiore.csv")
files.download("metrica2_ci_superiore.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>